# Domain-to-DSL OG-RAG Pipeline

Notebook này minh họa toàn bộ quy trình:

1. Chuyển đổi ontology mở rộng thành JSON-LD để thuật toán OG-RAG xử lý.
2. Khởi chạy DomainDSLQueryEngine để kết hợp tri thức domain + DSL.
3. Chạy thử với câu hỏi tự nhiên và xem DSL được sinh.
4. (Tuỳ chọn) Chạy batch câu hỏi từ file và ghi kết quả.

Prerequisites:
- Điền API key vào `api_keys.yaml`.
- Cài dependencies (`pip install -r requirements_minimal_hypergraph.txt`).


In [1]:
import os
import sys
import yaml
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

# Add project root to sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Working directory: {PROJECT_ROOT}")


Working directory: /media/thuongnv/New Volume/Code/Github/ograg2-1


## 0. Load API Keys

In [2]:
api_keys_file = "api_keys.yaml"

if not os.path.exists(api_keys_file):
    sample_config = {
        'GROQ_API_KEY': 'gsk_your-groq-key-here'
    }
    with open(api_keys_file, 'w') as f:
        yaml.dump(sample_config, f, default_flow_style=False)
    print("⚠️ Created api_keys.yaml - Please add your Groq API key!")
else:
    with open(api_keys_file, 'r') as f:
        api_keys = yaml.safe_load(f)
    
    # Set environment variables
    for key, value in api_keys.items():
        if value:
            os.environ[key] = str(value)
    
    print(f"✅ Loaded {len(api_keys)} API key(s) from {api_keys_file}")


✅ Loaded 1 API key(s) from api_keys.yaml


## 1. Chuyển đổi ontology mở rộng thành JSON-LD


In [3]:
!python prepare_domain_dsl_sample.py


Generated OG-RAG artifacts from the extended ontologies:
- Domain JSON-LD: /media/thuongnv/New Volume/Code/Github/ograg2-1/data/kg/domain_extended/ontology/domain_extended.jsonld
- DSL JSON-LD:    /media/thuongnv/New Volume/Code/Github/ograg2-1/data/kg/dsl_extended/ontology/dsl_extended.jsonld
- Rules file:     /media/thuongnv/New Volume/Code/Github/ograg2-1/data/rules/dsl_rules.txt

Run: python query_llm.py --config_file configs/rag/config_domain_dsl.yaml
Then provide natural language instructions to obtain DSL outputs.


Sau bước trên, các tệp sau được tạo/cập nhật:
- `data/kg/domain_extended/ontology/domain_extended.jsonld`
- `data/kg/dsl_extended/ontology/dsl_extended.jsonld`
- `data/rules/dsl_rules.txt`

Cấu hình `configs/rag/config_domain_dsl.yaml` đã trỏ tới các thư mục này.


## 2. Nạp config và khởi tạo DomainDSLQueryEngine


In [4]:
# Import minimal dependencies (không dùng utils.utils để tránh azureml)
import json
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings

# Load config manually
config_file = "configs/rag/config_domain_dsl.yaml"
if not os.path.exists(config_file):
    print(f"❌ Config file not found: {config_file}")
    print("Please run prepare_domain_dsl_sample.py first or check config path")
else:
    with open(config_file, 'r') as f:
        import yaml
        config_dict = yaml.safe_load(f)
    
    # Initialize LLM
    groq_api_key = os.environ.get('GROQ_API_KEY')
    if not groq_api_key:
        print("⚠️ GROQ_API_KEY not found in environment!")
    
    llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        api_key=groq_api_key,
        temperature=0.0,
        max_tokens=2048
    )
    
    # Initialize embedding model
    embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    
    print("✅ LLM initialized: llama-3.3-70b-versatile (Groq)")
    print("✅ Embedding model: sentence-transformers/all-MiniLM-L6-v2")
    print(f"\n📋 Config loaded from: {config_file}")


/home/thuongnv/.pyenv/versions/3.11.9/lib/python3.11/importlib/__init__.py:126: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  return _bootstrap._gcd_import(name[level:], package, level)
2025-11-12 15:02:18.275780: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-12 15:02:18

✅ LLM initialized: llama-3.3-70b-versatile (Groq)
✅ Embedding model: sentence-transformers/all-MiniLM-L6-v2

📋 Config loaded from: configs/rag/config_domain_dsl.yaml


## 2B. Initialize Query Engine (Simplified - No AzureML dependency)

In [5]:
# For now, use simple approach without full DomainDSLQueryEngine
# You can directly use the SPARQL-VI engine approach

print("""
⚠️ Note: DomainDSLQueryEngine requires full dependencies.

For SPARQL-VI OG-RAG, please use:
- test_sparql_vi_ograg_clean.ipynb (simple, working)
- test_hypergraph_original.ipynb (original OG-RAG)

Or install full dependencies:
    pip install -r requirements.txt
""")


⚠️ Note: DomainDSLQueryEngine requires full dependencies.

For SPARQL-VI OG-RAG, please use:
- test_sparql_vi_ograg_clean.ipynb (simple, working)
- test_hypergraph_original.ipynb (original OG-RAG)

Or install full dependencies:
    pip install -r requirements.txt



## Alternative: Use SPARQL-VI OG-RAG Engine (Recommended)

Để tránh dependency `azureml`, hãy dùng **SPARQL-VI OG-RAG Engine** đã được implement:

**File**: `test_sparql_vi_ograg_clean.ipynb`

**Features**:
- ✅ Two-Ontology Architecture (DSL + Domain)
- ✅ No AzureML dependency
- ✅ Simple, clean code
- ✅ Full English ontology
- ✅ Works with Groq LLM

In [6]:
# Import SPARQL-VI OG-RAG Engine (no azureml dependency)
import importlib.util
from pathlib import Path

# Load engine module directly
spec = importlib.util.spec_from_file_location(
    "sparql_vi_ograg_engine",
    "query_engine/sparql_vi_ograg_engine.py"
)
engine_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(engine_module)
SPARQLVIQueryEngine = engine_module.SPARQLVIQueryEngine

# Initialize SPARQL-VI engine
engine = SPARQLVIQueryEngine(
    dsl_facts_path="data/dsl/sparql_vi/dsl_facts.json",
    domain_facts_path="data/dsl/sparql_vi/domain_facts.json",
    test_cases_path="data/dsl/sparql_vi/test_cases.json",
    llm_api_key=os.environ.get('GROQ_API_KEY')
)

print("\n✅ SPARQL-VI OG-RAG Engine initialized successfully!")
print(f"   - DSL Hypergraph: {len(engine.dsl_hypergraph.nodes)} nodes, {len(engine.dsl_hypergraph.edges)} edges")
print(f"   - Domain Hypergraph: {len(engine.domain_hypergraph.nodes)} nodes, {len(engine.domain_hypergraph.edges)} edges")


FileNotFoundError: [Errno 2] No such file or directory: '/media/thuongnv/New Volume/Code/Github/ograg2-1/query_engine/sparql_vi_ograg_engine.py'

> **Lưu ý**: để `get_config()` lấy đúng file YAML khi chạy notebook độc lập, mở terminal và gọi:
> ```bash
> jupyter notebook notebooks/domain_dsl_pipeline.ipynb --NotebookApp.argv='["--config_file","configs/rag/config_domain_dsl.yaml"]'
> ```
> hoặc chỉnh `sys.argv` trước khi gọi `get_config()`. Khi chạy bên ngoài notebook, `query_llm.py` vẫn dùng đúng config.


## 3. Thử truy vấn tương tác


In [ ]:
sample_queries = [
    "Find all people over 18 years old",
    "Create friendship relations between people who live in the same city",
    "Count products in each category",
]

for query in sample_queries:
    print("#" * 80)
    print(f"Question: {query}\n")
    
    # Retrieve context
    context = engine.retrieve_context(
        query, 
        dsl_top_k=5,
        domain_top_k=10
    )
    
    print("📚 Retrieved Context (first 500 chars):")
    print(context[:500] + "...\n")
    
    # Generate query with OG-RAG
    vi_result = engine.generate_ograg(
        query, 
        dsl_top_k=5,
        domain_top_k=10
    )
    
    print(f"Generated SPARQL-VI:\n{vi_result}\n")
    
    # Validate
    validation = engine.validate_query(vi_result)
    print(f"✅ Syntax Valid: {validation['syntax_valid']}")
    if not validation['syntax_valid']:
        print(f"❌ Errors: {validation['errors']}")
    print()


## 4. Batch câu hỏi (tuỳ chọn)

Nếu `config.query.questions_file` chứa đường dẫn file, có thể chạy batch để ghi lại kết quả.


In [ ]:
from tqdm import tqdm
import time
import json
import pandas as pd

class QnAIO:
    def __init__(self):
        from collections import defaultdict
        self.data = defaultdict(list)

    def read(self, file_paths):
        if isinstance(file_paths, str):
            file_paths = [file_paths]
        for file_path in file_paths:
            with open(file_path, "r", encoding="utf-8") as f:
                if file_path.endswith(".json"):
                    entries = json.load(f)
                elif file_path.endswith(".csv"):
                    entries = pd.read_csv(f).to_dict("records")
                else:
                    raise ValueError("Unsupported format")
                for entry in entries:
                    for key, value in entry.items():
                        if key != "metadata":
                            self.data[key].append(value)

    def write(self, file_path, **kwargs):
        for key, value in kwargs.items():
            self.data[key] = value
        if file_path.endswith(".json"):
            records = [dict(zip(self.data, values)) for values in zip(*self.data.values())]
            with open(file_path, "w", encoding="utf-8") as f:
                json.dump(records, f, indent=2)
        elif file_path.endswith(".csv"):
            pd.DataFrame(self.data).to_csv(file_path, index=False)
        else:
            raise ValueError("Unsupported output format")

if config.query.questions_file:
    qa_io = QnAIO()
    qa_io.read(config.query.questions_file)
    questions = qa_io.data["question"]
    answers, contexts, elapsed = [], [], []
    for question in tqdm(questions):
        start = time.time()
        response, retrieved = domain_dsl_engine.query(
            query_str=question,
            domain_top_k=config.query.hyperparams.get("domain_top_k", 5),
            domain_nodes_top_k=config.query.hyperparams.get("domain_nodes_top_k", 30),
            dsl_top_k=config.query.hyperparams.get("dsl_top_k", 5),
            dsl_nodes_top_k=config.query.hyperparams.get("dsl_nodes_top_k", 30),
            return_context=True,
        )
        answers.append(response.response)
        contexts.append(retrieved)
        elapsed.append(time.time() - start)
    output_path = config.query.answers_file
    qa_io.write(output_path, answer=answers, retrieved_context=contexts, time=elapsed)
    print(f"Batch results saved to {output_path}")
else:
    print("No question files configured; skipping batch run.")


## 5. Đánh giá (optional)

Nếu cần sử dụng `test_answers.py` giống pipeline chuẩn, chạy cell dưới đây sau khi đã tạo file answers.


In [ ]:
# !python test_answers.py --config_file configs/rag/config_domain_dsl.yaml


Notebook hoàn tất. Bạn có thể chỉnh sửa prompt, luật hoặc bật vector index nếu muốn mở rộng.
